# Ch 31. 서빙 — llama.cpp server · vLLM · 지연 예산

Part 6 의 GGUF 를 진짜 서비스로: 3가지 서빙 스택 비교, 지연 예산 산수, 배치·동시성, 헬스체크·그레이스풀 셧다운·어댑터 핫스왑.

In [ ]:
# !pip install -q openai httpx asyncio

import asyncio
import time
import random

print("환경 준비 완료")
print("주의: llama-server 또는 vLLM 이 실행 중이어야 실제 요청 가능")

## 최소 예제 — 지연 예산 산수

```
지연 = prefill_time + decode_time
prefill_time = (입력 토큰 수) / prefill_TPS
decode_time  = (출력 토큰 수) / decode_TPS
```

본 책 모델 (Qwen 0.5B Q4, M2 Pro) 기준으로 계산.

In [ ]:
def estimate_latency(n_input, n_output,
                     prefill_tps=2000, decode_tps=150):
    """지연 예산 산수 (Qwen 0.5B Q4 M2 Pro 기준)."""
    prefill_time = n_input / prefill_tps
    decode_time  = n_output / decode_tps
    total = prefill_time + decode_time
    return prefill_time, decode_time, total

# 입력 200 토큰, 출력 100 토큰
n_in, n_out = 200, 100
pre, dec, total = estimate_latency(n_in, n_out)

print(f"입력 {n_in} 토큰 → prefill: {pre:.2f}s")
print(f"출력 {n_out} 토큰 → decode:  {dec:.2f}s")
print(f"합계: {total:.2f}s")
print()

# 다양한 시나리오
scenarios = [(50, 50), (200, 100), (500, 200), (1000, 500)]
budget = 1.5  # p95 예산 (초)
print(f"{'입력':>6} {'출력':>6} {'지연(s)':>10} {'p95 예산 1.5s':>15}")
print("-" * 45)
for ni, no in scenarios:
    _, _, t = estimate_latency(ni, no)
    ok = "OK" if t <= budget else "초과"
    print(f"{ni:>6} {no:>6} {t:>10.2f} {ok:>15}")

## 최소 예제 — llama.cpp server (OpenAI 호환 API)

llama-server 실행 명령 및 Python 클라이언트 코드.

In [ ]:
# llama-server 실행 방법 (터미널에서):
# ./llama.cpp/llama-server \
#     -m dist/tiny-tale-q4km.gguf \
#     --host 0.0.0.0 --port 8080 \
#     --ctx-size 1024 \
#     --threads 8 \
#     --n-gpu-layers -1      # Apple Silicon Metal

# OpenAI 호환 클라이언트 사용:
# from openai import OpenAI
# client = OpenAI(base_url="http://localhost:8080/v1", api_key="dummy")
# resp = client.chat.completions.create(
#     model="tiny-tale",
#     messages=[{"role": "user", "content": "옛날 옛적에"}],
#     temperature=0.8, max_tokens=120,
# )
# print(resp.choices[0].message.content)

print("llama-server 가 localhost:8080 에서 실행 중이어야 위 코드가 동작합니다.")
print("장점: 표준 OpenAI 클라이언트 그대로. 마이그레이션 부담 0.")

## 실전 — 동시성 부하 테스트

단일 요청 baseline vs 동시 10 vs 동시 50 의 p50/p95 측정.

In [ ]:
# !pip install -q httpx

# 실제 서버가 실행 중일 때 사용:
# import asyncio, httpx, time
#
# async def request(client, prompt):
#     r = await client.post("http://localhost:8000/v1/chat/completions", json={
#         "model": "qwen", "messages": [{"role": "user", "content": prompt}],
#         "max_tokens": 50,
#     })
#     return r.json()
#
# async def benchmark(n_concurrent):
#     async with httpx.AsyncClient() as client:
#         t0 = time.time()
#         results = await asyncio.gather(*[request(client, "테스트") for _ in range(n_concurrent)])
#         dt = time.time() - t0
#         print(f"  동시 {n_concurrent}: {dt:.2f}s, p95 ~{dt*0.95:.2f}s")
#
# asyncio.run(benchmark(1))    # baseline
# asyncio.run(benchmark(10))   # 동시성 영향
# asyncio.run(benchmark(50))   # 처리량 한계

# 지연 시뮬레이션 (서버 없이 개념 확인)
def simulate_latency(n_concurrent, base_latency=0.8, saturation=10):
    """동시 사용자 수에 따른 지연 시뮬레이션."""
    # 큐잉 이론: Little's law 근사
    if n_concurrent <= saturation:
        return base_latency * (1 + n_concurrent / saturation * 0.3)
    else:
        return base_latency * (1 + 0.3 + (n_concurrent - saturation) / saturation * 1.5)

print(f"{'동시 사용자':>12} {'예상 p50 (s)':>14} {'예상 p95 (s)':>14}")
print("-" * 45)
for n in [1, 5, 10, 20, 50, 100]:
    p50 = simulate_latency(n)
    p95 = p50 * 1.3
    print(f"{n:>12} {p50:>14.2f} {p95:>14.2f}")

## 실전 — 헬스체크 · vLLM LoRA 어댑터 핫스왑

In [ ]:
# vLLM 실행 방법 (터미널에서):
# pip install vllm
# python -m vllm.entrypoints.openai.api_server \
#     --model Qwen/Qwen2.5-0.5B-Instruct \
#     --host 0.0.0.0 --port 8000 \
#     --max-model-len 4096 \
#     --gpu-memory-utilization 0.9

# 헬스체크 endpoint (FastAPI 예시):
# @app.get("/health")
# async def health():
#     try:
#         resp = await client.chat.completions.create(
#             model="qwen", messages=[{"role": "user", "content": "ping"}],
#             max_tokens=5, timeout=2.0)
#         return {"status": "ok"}
#     except:
#         return {"status": "unhealthy"}, 503

# LoRA 어댑터 핫스왑 (vLLM 에서):
# vLLM serve 시: --enable-lora --max-loras 4
# 추론 시 어댑터 지정:
# resp = client.chat.completions.create(
#     model="qwen-with-adapter-v2",   # 어댑터 이름
#     messages=[...]
# )
# → 롤백 30초 안 가능 (어댑터 swap만, base 재로드 X)

# 롤백 예시:
# client.set_lora("adapter_v1")   # vLLM API

print("서빙 스택 비교:")
stacks = [
    ("llama.cpp server", "CPU/Mac/Vulkan 모두 OK, 가벼움", "큰 모델·많은 동시 사용자 약함", "사내망·노트북·작은 서비스"),
    ("vLLM",            "PagedAttention, 처리량 최강",    "GPU 만, 무거움",               "GPU 한 장+ 많은 사용자"),
    ("HF TGI",          "표준 HF 모델 자동 지원",          "운영 학습곡선",                 "HF 의존 환경"),
    ("Ollama",          "사용자 친화",                    "운영급 X",                     "데모·개발"),
]
print(f"{'스택':<18} {'강점':<28} {'약점':<22} {'어디'}")
print("-" * 90)
for s in stacks:
    print(f"{s[0]:<18} {s[1]:<28} {s[2]:<22} {s[3]}")

## 연습문제

1. 본 책 GGUF 모델을 `llama-server` 로 띄워 OpenAI SDK 로 호출.
2. 단일 요청 vs 동시 10 vs 동시 50 의 p50/p95 측정.
3. vLLM (가능하면) 으로 같은 부하 테스트. 처리량 차이.
4. 어댑터 2개를 vLLM 에 동시 로드 → 요청별 다른 어댑터 사용.
5. **(생각해볼 것)** AICC 콜 마감 후 1초 안에 요약 — p95 1초 예산이라면 어느 스택?

In [ ]:
# 연습 1: llama-server 로 모델 띄우기 + OpenAI SDK 호출


In [ ]:
# 연습 2: 동시 1 / 10 / 50 부하 테스트 p50/p95 측정


In [ ]:
# 연습 3: vLLM 부하 테스트 (GPU 있는 경우)


In [ ]:
# 연습 4: vLLM LoRA 어댑터 2개 동시 로드 + 요청별 다른 어댑터
